In [1]:
! which python 

/home/zhanyu/anaconda3/envs/kaggle/bin/python


# Kaggle房价预测

## 实现几个函数来方便下载数据

In [2]:
import hashlib  # 导入哈希库，用于生成文件的SHA-1哈希值
import os       # 导入os库，用于文件路径和目录操作
import tarfile  # 导入tarfile库，用于解压tar格式文件
import zipfile  # 导入zipfile库，用于解压zip格式文件
import requests # 导入requests库，用于发送HTTP请求下载文件

# 定义一个字典，用于存储数据集名称与对应的URL及哈希值
DATA_HUB = dict()

# 数据集的基础URL，用于下载数据
DATA_URL = 'http://d2l-data.s3-accelerate.amazonaws.com/'

def download(name, cache_dir=os.path.join('..', 'data')):  
    """下载一个DATA_HUB中的文件，返回本地文件名。"""
    # 确保要下载的文件名称在DATA_HUB字典中，否则报错
    assert name in DATA_HUB, f"{name} 不存在于 {DATA_HUB}."
    
    # 获取文件的URL和对应的SHA-1哈希值
    url, sha1_hash = DATA_HUB[name]
    
    # 创建缓存目录（如不存在），用于存放下载的文件
    os.makedirs(cache_dir, exist_ok=True)
    
    # 生成文件的本地路径，文件名取自URL的最后一部分
    fname = os.path.join(cache_dir, url.split('/')[-1])
    
    # 如果文件已经存在于缓存目录中，校验其哈希值以确定文件完整性
    if os.path.exists(fname):
        sha1 = hashlib.sha1()  # 创建一个SHA-1哈希对象
        with open(fname, 'rb') as f:
            # 逐块读取文件数据，避免一次性读入内存
            while True:
                data = f.read(1048576)  # 每次读取1MB的数据块
                if not data:  # 如果没有数据可读，说明文件已读取完毕
                    break
                sha1.update(data)  # 更新哈希对象的数据
        # 如果本地文件的哈希值与DATA_HUB中记录的值相同，返回文件路径
        if sha1.hexdigest() == sha1_hash:
            return fname
    
    # 如果文件不存在或哈希校验未通过，重新下载文件
    print(f'正在从{url}下载{fname}...')
    # 使用requests发送GET请求以流式方式下载文件
    r = requests.get(url, stream=True, verify=True)
    # 以二进制写模式打开文件，并将下载内容写入文件
    with open(fname, 'wb') as f:
        f.write(r.content)  # 将响应内容写入文件
    return fname  # 返回本地文件路径

## 访问和读取数据集

In [3]:
%matplotlib inline 

import numpy as np  # 导入NumPy库，用于数组和矩阵计算
import pandas as pd  # 导入Pandas库，用于数据处理和分析
import torch  # 导入PyTorch库，用于构建和训练深度学习模型
from torch import nn  # 从torch中导入神经网络模块
from d2l import torch as d2l  # 从d2l库中导入PyTorch部分，提供深度学习工具

# 将训练数据文件和测试数据文件的URL及其SHA-1哈希值存储在DATA_HUB字典中
DATA_HUB['kaggle_house_train'] = (  
    DATA_URL + 'kaggle_house_pred_train.csv',  # 训练数据的下载链接
    '585e9cc93e70b39160e7921475f9bcd7d31219ce')  # 训练数据的SHA-1哈希值，用于校验文件完整性

DATA_HUB['kaggle_house_test'] = (  
    DATA_URL + 'kaggle_house_pred_test.csv',  # 测试数据的下载链接
    'fa19780a7b011d9b009e8bff8e99922a8ee2eb90')  # 测试数据的SHA-1哈希值，用于校验文件完整性

# 下载训练数据并加载为Pandas数据框
train_data = pd.read_csv(download('kaggle_house_train'))

# 下载测试数据并加载为Pandas数据框
test_data = pd.read_csv(download('kaggle_house_test'))

# 打印训练数据和测试数据的形状（行数和列数），用于了解数据集大小
# 训练数据集包括1460个样本，每个样本80个特征和1个标签， 而测试数据集包含1459个样本，每个样本80个特征
print(train_data.shape)
print(test_data.shape)


(1460, 81)
(1459, 80)


In [4]:
# 让我们看看前四个和最后两个特征，以及相应标签（房价）。
print(train_data.iloc[0:4, [0, 1, 2, 3, -3, -2, -1]])

   Id  MSSubClass MSZoning  LotFrontage SaleType SaleCondition  SalePrice
0   1          60       RL         65.0       WD        Normal     208500
1   2          20       RL         80.0       WD        Normal     181500
2   3          60       RL         68.0       WD        Normal     223500
3   4          70       RL         60.0       WD       Abnorml     140000


In [5]:
# 在每个样本中，第一个特征是ID， 这有助于模型识别每个训练样本。 虽然这很方便，但它不携带任何用于预测的信息。 因此，在将数据提供给模型之前，我们将其从数据集中删除
all_features = pd.concat((train_data.iloc[:, 1:-1], test_data.iloc[:, 1:]))
# train_data.iloc[:, 1:-1]：删除了ID列（第一列）和房价标签列（最后一列）后的训练集数据。
# test_data.iloc[:, 1:]：删除了ID列后的测试集数据。
# pd.concat()：将处理后的训练集和测试集特征数据按列合并，得到一个新的 all_features 数据框。

## 数据预处理

In [6]:
# 将所有缺失的值替换为相应特征的平均值。 通过将特征重新缩放到零均值和单位方差来标准化数据
# 选择所有数值类型的特征（即不包括类别型特征）
numeric_features = all_features.dtypes[all_features.dtypes != 'object'].index

# 对所有数值型特征进行标准化，即将每个特征的均值变为0，标准差变为1
# 使用apply函数对每个数值型特征列进行操作，lambda表达式表示每列数据减去均值并除以标准差
all_features[numeric_features] = all_features[numeric_features].apply(
    lambda x: (x - x.mean()) / (x.std()))
    
# 对所有数值型特征进行缺失值填充，填充的值是该特征列的均值（0）
all_features[numeric_features] = all_features[numeric_features].fillna(0)


In [7]:
# 处理离散值。我们用一次独热编码替换它们
# pd.get_dummies() 用于将所有类别型（离散型）特征进行独热编码。
# 参数 dummy_na=True 表示将缺失值（NaN）也转换为一个新的列，表示是否为缺失值。
all_features = pd.get_dummies(all_features, dummy_na=True)

# 输出转换后的数据框的形状（行数和列数），查看独热编码后的结果
all_features.shape

(2919, 330)

In [10]:
# 从pandas格式中提取NumPy格式，并将其转换为张量表示
# 获取训练集的样本数（行数）
n_train = train_data.shape[0]

# 将训练集特征转换为PyTorch张量，dtype=torch.float32表示数据类型为32位浮点数
train_features = torch.tensor(all_features[:n_train].values.astype(float),
                              dtype=torch.float32)  # all_features中的前n_train行，作为训练特征

# 将测试集特征转换为PyTorch张量，dtype=torch.float32表示数据类型为32位浮点数
test_features = torch.tensor(all_features[n_train:].values.astype(float), 
                             dtype=torch.float32)  # all_features中的从n_train到结束的行，作为测试特征

# 将训练集标签（房价）转换为PyTorch张量，reshape(-1, 1)使标签成为一个列向量
train_labels = torch.tensor(train_data.SalePrice.values.astype(float).reshape(-1, 1), 
                            dtype=torch.float32)  # 将训练集中的房价（SalePrice）作为训练标签
